In [13]:
import pandas as pd
import numpy as np

column_names = ["Id", "Name", "Age", "Weight", "m0006", "m0612", "m1218", "f0006", "f0612", "f1218"]
df = pd.read_csv("patient_heart_rate.csv", header=None, names=column_names)
df.head()

,Id,Name,Age,Weight,m0006,m0612,m1218,f0006,f0612,f1218
0,John,Doe,30.0,70kg,m0006,75.0,f1218,80.0,NaN,NaN
1,Jane,Smith,25.0,132lbs,m0006,NaN,f1218,72.0,NaN,NaN
2,John,Doe,30.0,70kg,m0006,75.0,f1218,80.0,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Ali,Bábá,NaN,65kg,m0006,60.0,f1218,NaN,NaN,NaN


In [14]:
def split_name(name_val):
    if pd.isna(name_val):
        return pd.Series([np.nan, np.nan])
    name_str = str(name_val).strip()
    parts = name_str.split(maxsplit=1)
    if len(parts) == 2:
        return pd.Series([parts[0], parts[1]])
    elif len(parts) == 1:
        return pd.Series([parts[0], ""])
    else:
        return pd.Series([np.nan, np.nan])

df[['Firstname', 'Lastname']] = df['Name'].apply(split_name)
df = df.drop('Name', axis=1)
df.head()

,Id,Age,Weight,m0006,m0612,m1218,f0006,f0612,f1218,Firstname,Lastname
0,John,30.0,70kg,m0006,75.0,f1218,80.0,NaN,NaN,Doe,
1,Jane,25.0,132lbs,m0006,NaN,f1218,72.0,NaN,NaN,Smith,
2,John,30.0,70kg,m0006,75.0,f1218,80.0,NaN,NaN,Doe,
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Ali,NaN,65kg,m0006,60.0,f1218,NaN,NaN,NaN,Bábá,


In [15]:
def clean_and_convert_weight(weight_val):
    if pd.isna(weight_val):
        return np.nan
    w_str = str(weight_val).lower().strip()
    if 'lbs' in w_str:
        val = float(w_str.replace('lbs', ''))
        return round(val / 2.2, 1)
    elif 'kgs' in w_str:
        return float(w_str.replace('kgs', ''))
    elif 'kg' in w_str:
        return float(w_str.replace('kg', ''))
    else:
        try:
            return float(w_str)
        except ValueError:
            return np.nan

df['Weight'] = df['Weight'].apply(clean_and_convert_weight)
df.head()

,Id,Age,Weight,m0006,m0612,m1218,f0006,f0612,f1218,Firstname,Lastname
0,John,30.0,70.0,m0006,75.0,f1218,80.0,NaN,NaN,Doe,
1,Jane,25.0,60.0,m0006,NaN,f1218,72.0,NaN,NaN,Smith,
2,John,30.0,70.0,m0006,75.0,f1218,80.0,NaN,NaN,Doe,
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Ali,NaN,65.0,m0006,60.0,f1218,NaN,NaN,NaN,Bábá,


In [16]:
df = df.dropna(how='all')
df = df.drop_duplicates(subset=['Firstname', 'Lastname', 'Age', 'Weight'])
df.head()

,Id,Age,Weight,m0006,m0612,m1218,f0006,f0612,f1218,Firstname,Lastname
0,John,30.0,70.0,m0006,75.0,f1218,80.0,NaN,NaN,Doe,
1,Jane,25.0,60.0,m0006,NaN,f1218,72.0,NaN,NaN,Smith,
4,Ali,NaN,65.0,m0006,60.0,f1218,NaN,NaN,NaN,Bábá,
5,Elena,40.0,NaN,m0006,85.0,f1218,88.0,NaN,NaN,Russo,
6,Tom,NaN,NaN,m0006,NaN,f1218,NaN,NaN,NaN,Hanks,


In [17]:
df['Firstname'] = df['Firstname'].replace({r'[^\x00-\x7F]+': ''}, regex=True)
df['Lastname'] = df['Lastname'].replace({r'[^\x00-\x7F]+': ''}, regex=True)
df.head()

,Id,Age,Weight,m0006,m0612,m1218,f0006,f0612,f1218,Firstname,Lastname
0,John,30.0,70.0,m0006,75.0,f1218,80.0,NaN,NaN,Doe,
1,Jane,25.0,60.0,m0006,NaN,f1218,72.0,NaN,NaN,Smith,
4,Ali,NaN,65.0,m0006,60.0,f1218,NaN,NaN,NaN,Bb,
5,Elena,40.0,NaN,m0006,85.0,f1218,88.0,NaN,NaN,Russo,
6,Tom,NaN,NaN,m0006,NaN,f1218,NaN,NaN,NaN,Hanks,


In [18]:
print("--- SỐ LƯỢNG DỮ LIỆU THIẾU BAN ĐẦU ---")
print(df[['Age', 'Weight']].isna().sum())
print("-" * 40)

df = df.dropna(subset=['Age', 'Weight'], how='all')
mean_age = df['Age'].mean()
df['Age'] = df['Age'].fillna(mean_age)

print("--- THỐNG KÊ SAU KHI ĐIỀN KHUYẾT ---")
print(df[['Age', 'Weight']].isna().sum())

--- SỐ LƯỢNG DỮ LIỆU THIẾU BAN ĐẦU ---
Age       2
Weight    2
dtype: int64
----------------------------------------
--- THỐNG KÊ SAU KHI ĐIỀN KHUYẾT ---
Age       0
Weight    1
dtype: int64


In [19]:
id_vars = ['Id', 'Age', 'Weight', 'Firstname', 'Lastname']
value_vars = ['m0006', 'm0612', 'm1218', 'f0006', 'f0612', 'f1218']

df_melted = pd.melt(df, id_vars=id_vars, value_vars=value_vars, var_name='sex_and_time', value_name='PulseRate')
df_melted['Sex'] = df_melted['sex_and_time'].str[0]
df_melted['Time'] = df_melted['sex_and_time'].str[1:3] + '-' + df_melted['sex_and_time'].str[3:]
df_melted = df_melted.drop(columns=['sex_and_time'])
df_melted.head()

,Id,Age,Weight,Firstname,Lastname,PulseRate,Sex,Time
0,John,30.000000,70.0,Doe,,m0006,m,00-06
1,Jane,25.000000,60.0,Smith,,m0006,m,00-06
2,Ali,31.666667,65.0,Bb,,m0006,m,00-06
3,Elena,40.000000,NaN,Russo,,m0006,m,00-06
4,John,30.000000,70.0,Doe,,75.0,m,06-12


In [20]:
df_melted['PulseRate'] = pd.to_numeric(df_melted['PulseRate'], errors='coerce')
df_melted = df_melted.sort_values(by=['Id', 'Time']).reset_index(drop=True)

missing_rate = df_melted['PulseRate'].isna().mean() * 100
print(f"Tỉ lệ dữ liệu thiếu trên biến huyết áp: {missing_rate:.2f}%")

def medical_fallback_fill(s):
    step1 = (s.shift(1) + s.shift(-1)) / 2
    step2 = (s.shift(1) + s.shift(2)) / 2
    step3 = (s.shift(-1) + s.shift(-2)) / 2
    step4 = s.mean()
    s = s.fillna(step1).fillna(step2).fillna(step3)
    if not pd.isna(step4):
        s = s.fillna(step4)
    return s

df_melted['PulseRate'] = df_melted.groupby('Id', group_keys=False)['PulseRate'].apply(medical_fallback_fill)

sex_mean = df_melted.groupby('Sex')['PulseRate'].transform('mean')
df_melted['PulseRate'] = df_melted['PulseRate'].fillna(sex_mean)

global_mean = df_melted['PulseRate'].mean()
default_value = global_mean if not pd.isna(global_mean) else 72.0
df_melted['PulseRate'] = df_melted['PulseRate'].fillna(default_value)

print("Đã hoàn tất xử lý điền khuyết chuỗi dữ liệu huyết áp thành công!")

Tỉ lệ dữ liệu thiếu trên biến huyết áp: 75.00%
Đã hoàn tất xử lý điền khuyết chuỗi dữ liệu huyết áp thành công!


In [21]:
ordered_columns = ['Id', 'Age', 'Weight', 'Firstname', 'Lastname', 'PulseRate', 'Sex', 'Time']
df_final = df_melted[ordered_columns].copy()
df_final = df_final.sort_values(by=['Id', 'Time']).reset_index(drop=True)
df_final.to_csv('patient_heart_rate_clean.csv', index=False)

print("--- FILE KẾT QUẢ CUỐI CÙNG ---")
df_final

--- FILE KẾT QUẢ CUỐI CÙNG ---


,Id,Age,Weight,Firstname,Lastname,PulseRate,Sex,Time
0,Ali,31.666667,65.0,Bb,,60.0,m,00-06
1,Ali,31.666667,65.0,Bb,,60.0,f,00-06
2,Ali,31.666667,65.0,Bb,,60.0,m,06-12
3,Ali,31.666667,65.0,Bb,,60.0,f,06-12
4,Ali,31.666667,65.0,Bb,,60.0,m,12-18
5,Ali,31.666667,65.0,Bb,,60.0,f,12-18
6,Elena,40.000000,NaN,Russo,,86.5,m,00-06
7,Elena,40.000000,NaN,Russo,,88.0,f,00-06
8,Elena,40.000000,NaN,Russo,,85.0,m,06-12
9,Elena,40.000000,NaN,Russo,,86.5,f,06-12
